# 17) Reconstructions and slope/flux limiters

* Notes on unstructured meshing workflow
* Finite volume methods for hyperbolic conservation laws
* Riemann solvers for scalar equations
  * Shocks and the Rankine-Hugoniot condition
  * Rarefactions and entropy solutions

Today:
1. Godunov's Theorem  
2. Slope reconstruction and limiting  
  2.1 Common limiters in the FV literature  
3. Introduction on hyperbolic systems  
  3.1 Example: Isentropic gas dynamics  

In [ ]:
using LinearAlgebra
using Plots
default(linewidth=3)

# utility functions for Runge-Kutta ODE time-steppers
struct RKTable
    A::Matrix
    b::Vector
    c::Vector
    function RKTable(A, b)
        s = length(b)
        A = reshape(A, s, s)
        c = vec(sum(A, dims=2))
        new(A, b, c)
    end
end

rk4 = RKTable([0 0 0 0; .5 0 0 0; 0 .5 0 0; 0 0 1 0], [1, 2, 2, 1] / 6)

function ode_rk_explicit(f, u0; tfinal=1., h=0.1, table=rk4)
    u = copy(u0)
    t = 0.
    n, s = length(u), length(table.c)
    fY = zeros(n, s)
    thist = [t]
    uhist = [u0]
    while t < tfinal
        tnext = min(t+h, tfinal)
        h = tnext - t
        for i in 1:s
            ti = t + h * table.c[i]
            Yi = u + h * sum(fY[:,1:i-1] * table.A[i,1:i-1], dims=2)
            fY[:,i] = f(ti, Yi)
        end
        u += h * fY * table.b
        t = tnext
        push!(thist, t)
        push!(uhist, u)
    end
    thist, hcat(uhist...)
end

# utility functions for Riemann problems and FV
function testfunc(x)
    max(1 - 4*abs.(x+2/3),
        abs.(x) .< .2,
        (2*abs.(x-2/3) .< .5) * cospi(2*(x-2/3)).^2
    )
end

flux_advection(u) = u
flux_burgers(u) = u^2/2
flux_traffic(u) = u * (1 - u)

riemann_advection(uL, uR) = 1*uL # velocity is +1

function fv_solve1(riemann, u_init, n, tfinal=1)
    h = 2 / n
    x = LinRange(-1+h/2, 1-h/2, n) # cell midpoints (centroids)
    idxL = 1 .+ (n-1:2*n-2) .% n
    idxR = 1 .+ (n+1:2*n) .% n
    function rhs(t, u)
        fluxL = riemann(u[idxL], u)
        fluxR = riemann(u, u[idxR])
        (fluxL - fluxR) / h
    end
    thist, uhist = ode_rk_explicit(rhs, u_init.(x), h=h, tfinal=tfinal)
    x, thist, uhist
end

function riemann_burgers(uL, uR)
    flux = zero(uL)
    for i in 1:length(flux)
        fL = flux_burgers(uL[i])
        fR = flux_burgers(uR[i])
        flux[i] = if uL[i] > uR[i] # shock
            max(fL, fR)
        elseif uL[i] > 0 # rarefaction all to the right
            fL
        elseif uR[i] < 0 # rarefaction all to the left
            fR
        else
            0
        end
    end
    flux
end

function riemann_traffic(uL, uR)
    flux = zero(uL)
    for i in 1:length(flux)
        fL = flux_traffic(uL[i])
        fR = flux_traffic(uR[i])
        flux[i] = if uL[i] < uR[i] # shock
            min(fL, fR)
        elseif uL[i] < .5 # rarefaction all to the right
            fL
        elseif uR[i] > .5 # rarefaction all to the left
            fR
        else
            flux_traffic(.5)
        end
    end
    flux 
end

Recall from last lecture that the Burger's equation has a nonlinear flux, so we found shock waves and rarefaction waves.

In [ ]:
init_func(x) = testfunc(x) - .5
x, thist, uhist = fv_solve1(riemann_burgers, init_func, 200, 1)
plot(x, uhist[:,1:20:end], legend=:none)

## Recap

Some examples seen so far:

### Burgers

* flux $u^2/2$ has speed $u$
* negative values make sense
* satisfies a maximum principle

### Traffic

* flux $u - u^2$ has speed $1 - 2u$
* state must satisfy $u \in [0, 1]$

## 1. Godunov's Theorem (1954)

Linear numerical methods
$$ \dot u_i = \sum_j a_{ij} u_j $$
for solving time-dependent PDE can be at most first order accurate if they are monotone.

For our purposes, monotonicity is equivalent to positivity preservation,
$$ \min_x u(0, x) \ge 0 \quad \Longrightarrow \quad \min_x u(t, 0) \ge 0 .$$

### Discontinuities

A numerical method for representing a discontinuous function on a stationary grid can be no better than first order accurate in the $L^1$ norm,
$$ \lVert u - u^* \rVert_{L^1} = \int \lvert u - u^* \rvert . $$
If we merely sample a discontinuous function, say
$$ u(x) = \begin{cases} 0, & x \le a \\ 1, & x > a \end{cases} $$
onto a grid with element size $\Delta x$ then we will have errors of order 1 on an interval proportional to $\Delta x$.

In light of these two observations, we may still ask for numerical methods that are more than first order accurate for smooth solutions, but those methods must be nonlinear.

## 2. Slope reconstruction and limiting 

> For many finite-difference or finite-volume computations with discontinuous solutions limiters are a _necessary evil_. NASA's Technical Report by [Berger, Aftosmis, and Murman (2005)](https://www.nas.nasa.gov/assets/nas/pdf/papers/nas-05-007.pdf)

One method for constructing higher order methods is to use the state in neighboring elements to perform a conservative reconstruction of a piecewise polynomial, then compute numerical fluxes by solving Riemann problems at the interfaces. 

If $x_i$ is the center of cell $i$ and $g_i$ is the reconstructed gradient inside cell $i$, our reconstructed solution is
$$ \tilde u_i(x) = u_i + g_i \cdot (x - x_i) . $$
We would like this reconstruction to be monotone in the sense that
$$ \Big(\tilde u_i(x) - \tilde u_j(x) \Big) \Big( u_i - u_j \Big) \ge 0 $$
for any $x$ on the interface between element $i$ and element $j$.

### Question
Is the symmetric centered difference slope
$$ \hat g_i = \frac{u_{i+1} - u_{i-1}}{2 \Delta x} $$
monotone?

### Slope limiting

We will determine gradients by "limiting" the above slope using a nonlinear function that reduces to 1 when the solution is smooth. 

There are many ways to express **[flux/slope limiters](https://en.wikipedia.org/wiki/Flux_limiter)** and our discussion here roughly follows this NASA's Technical Report by[Berger, Aftosmis, and Murman (2005)](https://www.nas.nasa.gov/assets/nas/pdf/papers/nas-05-007.pdf).

![Berger et al, figure 3b, Cell averages](../img/Berger2005-3b-CellAverages.png)

The figure above depicts cell averages corresponding to the limiters. When $u_i$ is exactly in the middle, the data is linear, $f = 1/2$, and the limiter is $1$. The undivided differences are $\Delta_{-} = u_i − u_{i−1}$ , and $\Delta_+ = u_{i+1} − u_i$. $J$ is the total jump $u_{i+1} − u_{i−1}$.

### Flux-corrected Transport (FCT) [Zalesak 1979](https://www.sciencedirect.com/science/article/abs/pii/0021999179900512)

Zalesak proposed a linear combination of low-order (think 1st order) and high-order (think 3rd-order) upwinding. For example, one can write a second-order scheme as a first-order scheme plus a _limited_ _anti-diffusive flux_, shown here for a linear advection/transport equation

$$
u_t + a u_x = 0, \qquad a> 0
$$

Zalesak's proposed modified scheme is
$$
u_i^{n+1} = u_i^n - \lambda (u_i - u_{i-1}) - (\phi_i F_{i+1/2} - \phi_{i-1} F_{i-1/2})
$$

where the anti-diffusive flux is $F_{i+1/2} = \frac{\lambda}{2}  (1-\lambda) (u_{i+1} -u_i)$. Here the $\phi$'s are the **flux limiters** and $\lambda = a \frac{\Delta t}{\Delta x}$.

We will express a slope limiter in terms of the ratio of backward to forward differences in the solution 
$$ r_i = \frac{u_i - u_{i-1}}{u_{i+1} - u_{i-1}} $$
and use as a gradient the reconstructed cell slope,
$$ g_i = \phi(r_i) \frac{u_{i+1} - u_{i-1}}{2 \Delta x} . $$


Our functions $\phi$ will be zero unless $0 < r < 1$ and $\phi(1/2) = 1$ (which makes the reconstruction exact for a linear profile).

- The main idea is use the centered finite difference scheme for the slope and multiply it by a nonlinear factor $\phi (r_i)$ which detects whether the local data are smooth and monotone.

- If the solution is locally linear, then we have the approximation

$$
u_i - u_{i-1} \approx \frac{u_{i+1} - u_{i-1}}{2}
$$

so, in this case, $r_i \approx 1/2$. That is why the limiter is normalized so that $\phi(1/2) = 1$.

- If $r_i$ is outside $(0,1)$, then $u_i$ is not between its neighbors, which is a warning sign for an extremum or non-monotone data, so the limiter cuts the slope down or all the way to zero.
    * This is the mechanism that avoids Gibbs-like oscillations near shocks.

![Berger et al, figure 3a, Symmetric form of slope limiters](../img/Berger2005-3a-SymSlope.png)

All of these limiters are second order accurate and **[total variation diminishing (TVD)](https://en.wikipedia.org/wiki/Total_variation_diminishing)**; those that fall below "[minmod](https://www.annualreviews.org/content/journals/10.1146/annurev.fl.18.010186.002005)" are not second order accurate and those that are above Barth-Jesperson are not second order accurate, not TVD, or produce artifacts.

### 2.1 Common limiters in the FV literature  

Referring to the names used in the code below, we have:

#### `limit_zero`
$$
\phi(r) = 0 \, ,
$$

so the reconstructed slope is always zero:

$$
g_i = 0
$$

This gives a piecewise constant reconstruction in each cell, which is just the first-order [Godunov method](https://en.wikipedia.org/wiki/Godunov%27s_scheme). It is the most diffusive limiter.

#### `limit_none`

$$
\phi(r) = 1 \, ,
$$

so the reconstructed slope is:

$$
g_i = \frac{u_{i+1} - u_{i-1}}{ 2 \Delta x}
$$

which is the unlimited centered finite difference slope. It is second-order in smooth regions but can create under/overshoots.

#### `limit_minmod`

$$
\phi(r) = \textrm{max}( \textrm{min} (2r, 2(1-r)),0)
$$

This behaves like

$$
\phi(r) = 
\begin{cases}
0, & \textrm{ for } & r \leq 0 , r \geq 1\\
2 r & \textrm{ for } & 0 < r \leq 1/2\\
2 (1-r) & \textrm{ for } & 1/2  <r < 1\\
\end{cases}
$$
This creates a triangular shape with peak value $1$ at $r=1/2$. It is a conservative, fairly dissipative limiter: it allows a slope only when the cell average $u_i$ lies between its two neighbors, and even then it trims the slope aggressively as you move away from the linear case $r=1/2$.

#### `limit_sin`

$$
\phi(r) = 
\begin{cases}
\sin (\pi r), & \textrm{ for } 0 < r < 1\\
0, & \textrm{otherwise} 
\end{cases}
$$
smoother than the previous one with same extrema behavior.

#### `limit_vl`

$$
\phi(r) = \textrm{max} (4r (1-r), 0)
$$

Similar to the previous one, quadratic shape (parabola) transition instead of a trigonometric one.

#### `limit_bj`

$$
\phi(r) = \textrm{max} (0, \textrm{min}(1, 4r, 4 (1-r)))
$$

This grows linearly from zero, then plateaus at 1 through a broad middle region, then decreases linearly back to zero as $r \to 1$. It is more permissive than `minmod`: if the solution looks reasonably smooth and monotone, it keeps the full centered slope over a larger range of $r$-values.

In [ ]:
limit_zero(r) = 0 # piecewise constant in each cell
limit_none(r) = 1 # no limiter, centered diff
limit_minmod(r) = max(min(2*r, 2*(1-r)), 0)
limit_sin(r) = (0 < r && r < 1) * sinpi(r)
limit_vl(r) = max(4*r*(1-r), 0)
limit_bj(r) = max(0, min(1, 4*r, 4*(1-r)))
limiters = [limit_zero limit_none limit_minmod limit_sin limit_vl limit_bj];

In [ ]:
plot(limiters, label=limiters, xlims=(-.1, 1.1))

### A FV solver with flux/slope limiter

In [ ]:
function fv_solve2(riemann, u_init, n, tfinal=1, limit=limit_sin)
    h = 2 / n # 1D domain [-1,1]
    x = LinRange(-1+h/2, 1-h/2, n) # cell midpoints (centroids)
    idxL = 1 .+ (n-1:2*n-2) .% n
    idxR = 1 .+ (n+1:2*n) .% n
    function rhs(t, u)
        jump = u[idxR] - u[idxL]
        r = (u - u[idxL]) ./ jump
        r[isnan.(r)] .= 0
        g = limit.(r) .* jump / 2h
        fluxL = riemann(u[idxL] + g[idxL]*h/2, u - g*h/2)
        fluxR = fluxL[idxR]
        (fluxL - fluxR) / h
    end
    thist, uhist = ode_rk_explicit(
        rhs, u_init.(x), h=h, tfinal=tfinal)
    x, thist, uhist
end

In [ ]:
x, thist, uhist = fv_solve2(riemann_advection, testfunc, 200, 10,
    limit_sin)
plot(x, uhist[:,1:400:end], legend=:none)

## 3. Introduction on hyperbolic systems 

Systems of the form 

$$ \mathbf U_t + \nabla \cdot F(\mathbf  U) = 0 $$

It would be too good if we could safely reconstruct each component of the flux in the same way. 

But the system has **waves traveling at different speeds**. So we need to be more careful with reconstructions.

### 3.1 Example: Isentropic gas dynamics

\begin{align} U &= \begin{bmatrix} \rho \\ \rho u \end{bmatrix} \, ,& F(U) &= \begin{bmatrix} \rho u \\ \rho u^2 + p \end{bmatrix} \end{align}

| Variable | meaning |
|----------|--------|
| $\rho$ | density |
| $u$    | velocity |
| $\rho u$ | momentum |
| $p$      | pressure |


* Isentrtropic equation of state (pressure depends only on density) $p(\rho) = C \rho^\gamma$ with $\gamma = 1.4$ (typical air). 
* "isothermal" gas dynamics: $p(\rho) = c^2 \rho$, wave speed $c$.
* Compute the needed $\rho u^2$  as $ \rho u^2 = \frac{(\rho u)^2}{\rho} .$

The flux Jacobian is

$$
F'(U) = \frac{\partial F}{\partial U}
$$



#### Smooth wave structure

For perturbations of a constant state, systems of equations produce multiple waves with speeds equal to the eigenvalues of the flux Jacobian,
$$ F'(U) = \begin{bmatrix} 0 & 1 \\ -u^2 + c^2 & 2 u \end{bmatrix}. $$
We can compute the eigenvalues $\Lambda $ and eigenvector $W$ matrices,
$$ F'(U) = W \Lambda W^{-1} $$
as
\begin{align} 
W &= \begin{bmatrix} 1 & 1 \\ u-c & u+c \end{bmatrix} &
\Lambda &= \begin{bmatrix} u-c &  \\ & u+c \end{bmatrix} .
\end{align}